# Day 6: Session 6B - Long and Wide

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/6b_reshaping_data.html)

Date: 09/08/2026

In [1]:
import pandas as pd

base = 'https://eds-217-essential-python.github.io/data/'

goleta = pd.read_csv(base + 'openaq_goleta_measurments.csv')
santa_barbara = pd.read_csv(base + 'openaq_santa_barbara_measurments.csv')
cnsi = pd.read_csv(base + 'openaq_CNSI_measurments.csv')

print(goleta.shape)
print(santa_barbara.shape)
print(cnsi.shape)

(1962, 15)
(2266, 15)
(714, 15)


In [3]:
# can use pd.concat to stack dfs ontop of 
# each other if they have the same columns

aq = pd.concat([goleta, santa_barbara, cnsi])

aq.shape


(4942, 15)

In [3]:
# when stacking dfs you aren't able to tell where they
# they came from so you gotta label each df BEFORE using pd.ConnectionAbortedError

goleta['site'] = 'Goleta'
santa_barbara['site'] = 'Santa Barbara'
cnsi['site'] = 'CNSI'

aq = pd.concat([goleta, santa_barbara, cnsi], ignore_index=True)

aq.shape

#ignore_index = True makes it so that the index of each df is not brought
# over to the concat df

(4942, 16)

In [14]:
anotherstack = pd.concat([goleta, cnsi])
anotherstack.shape

(2676, 16)

In [ ]:
# now its time to pivot dfs from long to wide and back around
means = aq.pivot_table(index='site', columns='parameter', values='value')

means


,location_id,location_name,parameter,value,unit,datetimeUtc,datetimeLocal,timezone,latitude,longitude,country_iso,isMobile,isMonitor,owner_name,provider,site
0,1186,Goleta,o3,0.025,ppm,2024-07-12T01:00:00+00:00,2024-07-11T18:00:00-07:00,America/Los_Angeles,34.445301,-119.827797,NaN,NaN,NaN,Unknown Governmental Organization,AirNow,Goleta
1,1186,Goleta,o3,0.028,ppm,2024-07-12T02:00:00+00:00,2024-07-11T19:00:00-07:00,America/Los_Angeles,34.445301,-119.827797,NaN,NaN,NaN,Unknown Governmental Organization,AirNow,Goleta
2,1186,Goleta,o3,0.029,ppm,2024-07-12T03:00:00+00:00,2024-07-11T20:00:00-07:00,America/Los_Angeles,34.445301,-119.827797,NaN,NaN,NaN,Unknown Governmental Organization,AirNow,Goleta
3,1186,Goleta,o3,0.027,ppm,2024-07-12T04:00:00+00:00,2024-07-11T21:00:00-07:00,America/Los_Angeles,34.445301,-119.827797,NaN,NaN,NaN,Unknown Governmental Organization,AirNow,Goleta
4,1186,Goleta,o3,0.026,ppm,2024-07-12T05:00:00+00:00,2024-07-11T22:00:00-07:00,America/Los_Angeles,34.445301,-119.827797,NaN,NaN,NaN,Unknown Governmental Organization,AirNow,Goleta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4937,2812592,CNSI Roof Top,pm25,9.900,µg/m³,2024-08-11T20:00:00+00:00,2024-08-11T13:00:00-07:00,America/Los_Angeles,34.415560,-119.840250,NaN,NaN,NaN,Clarity,Clarity,CNSI
4938,2812592,CNSI Roof Top,pm25,9.100,µg/m³,2024-08-11T21:00:00+00:00,2024-08-11T14:00:00-07:00,America/Los_Angeles,34.415560,-119.840250,NaN,NaN,NaN,Clarity,Clarity,CNSI
4939,2812592,CNSI Roof Top,pm25,8.700,µg/m³,2024-08-11T22:00:00+00:00,2024-08-11T15:00:00-07:00,America/Los_Angeles,34.415560,-119.840250,NaN,NaN,NaN,Clarity,Clarity,CNSI
4940,2812592,CNSI Roof Top,pm25,8.000,µg/m³,2024-08-11T23:00:00+00:00,2024-08-11T16:00:00-07:00,America/Los_Angeles,34.415560,-119.840250,NaN,NaN,NaN,Clarity,Clarity,CNSI


In [ ]:
# by default, the pivot function will take the mean of all
# values that you are pivoting around and what not
# you can be specific about what function to use when pivoting
# by using the aggfunc= argument

aq.pivot_table(index='site', columns='parameter', values='value', aggfunc='count')

#the count argument gives you the count of how many cells fit into 
# that one pivoted cell

parameter,o3,pm10,pm25
site,,,
CNSI,NaN,NaN,714.0
Goleta,711.0,517.0,734.0
Santa Barbara,734.0,766.0,766.0


In [10]:
#once you create the pivot table the index becomes the 'index=' column
# if you want to get the index back as a normal column you can 
# use .reset_index()

flat = means.reset_index()

flat

parameter,site,o3,pm10,pm25
0,CNSI,NaN,NaN,6.083473
1,Goleta,0.022470,14.972921,6.480926
2,Santa Barbara,0.019822,17.563969,6.172324


In [ ]:
# now we can create a df of counts per site
counts = aq['site'].value_counts()
type(counts)

# but we gotta change that from a series to a df so we can merge it 
# into other dfs

counts = aq['site'].value_counts().reset_index()

type(counts)



pandas.core.frame.DataFrame

In [20]:
# now we can do the merge, but first we should give counts a
# good and unique column name

counts = counts.rename(columns={'count': 'n_readings'})

summary = pd.merge(flat, counts, on='site')
summary

,site,o3,pm10,pm25,n_readings
0,CNSI,NaN,NaN,6.083473,714
1,Goleta,0.022470,14.972921,6.480926,1962
2,Santa Barbara,0.019822,17.563969,6.172324,2266


In [ ]:
# building a table of count numbers per paremeter
countscounts = aq.pivot_table(index='parameter', columns='site', values='value', aggfunc='count')
parametercount = aq['parameter'].value_counts().reset_index()
mergedcounts = pd.merge(countscounts,parametercount,on = 'parameter')
mergedcounts.sort_values('count', ascending= False)





,parameter,CNSI,Goleta,Santa Barbara,count
2,pm25,714.0,734.0,766.0,2214
0,o3,NaN,711.0,734.0,1445
1,pm10,NaN,517.0,766.0,1283


In [33]:
# now we can easily find an average hourly polutant rate using everything

aq['hour'] = aq['datetimeLocal'].str[11:13]

pm25 = aq[aq['parameter'] == 'pm25'].copy()

hourly = pm25.pivot_table(index='hour',
  columns='site',
  values='value')

hourly.reset_index().round(2)

site,hour,CNSI,Goleta,Santa Barbara
0,00,5.86,5.10,5.59
1,01,5.84,9.07,5.47
2,02,6.08,4.83,5.44
3,03,5.99,5.30,5.38
4,04,5.60,3.90,4.62
5,05,5.63,3.47,4.41
6,06,5.90,4.10,4.09
7,07,6.03,5.58,4.00
8,08,5.96,6.42,6.06
9,09,6.08,7.42,6.94


In [4]:
# what if i wanted to do it using groupby?

aq.groupby(['site', 'parameter'])['value'].mean()

site           parameter
CNSI           pm25          6.083473
Goleta         o3            0.022470
               pm10         14.972921
               pm25          6.480926
Santa Barbara  o3            0.019822
               pm10         17.563969
               pm25          6.172324
Name: value, dtype: float64